In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map(center=[23.685, 90.3563])

region = ee.Geometry.Rectangle([90.98, 22.01, 91.06, 22.08])

map.addLayer(region, {}, "region")

map.centerObject(region, 12)

map

Map(center=[22.045001971601074, 91.02000000000034], controls=(WidgetControl(options=['position', 'transparent_…

In [3]:
image = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(region) \
    .filterDate('2023-01-01', '2023-03-01') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 5)) \
    .median() \
    .clip(region)

training_image = image.select(['B2', 'B3', 'B4', 'B8']) # Blue, Green, Red, NIR



In [4]:
#training the cluster

training_data = training_image.sample(
    region=region,
    scale=10,
    numPixels=5000
)
clusterer = ee.Clusterer.wekaKMeans(4).train(training_data)
#Weka K-Means: Earth Engine's built-in wekaKMeans algorithm is fully supported in the Python API.
result = training_image.cluster(clusterer)


In [5]:
map1 = geemap.Map()
map1.add_basemap('HYBRID')

visParams = {
    'min': 0,
    'max': 3,
    'palette': [
        "#1f77b4",  # blue
        "#ff7f0e",  # orange
        "#2ca02c",  # green
        "#d62728",  # red
        "#9467bd"   # purple (extra color in case clusters shift)
    ]
}

map1.addLayer(result, visParams, "clusters")
map1.centerObject(region, 12)
map1


Map(center=[22.045001971601074, 91.02000000000034], controls=(WidgetControl(options=['position', 'transparent_…